# **Model Training - WhatsApp Spam & Fraud Risk Classification**

In [138]:
# 1. Imports

import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In [139]:
df = pd.read_csv('data/data.csv')
df = df.dropna(subset=['body', 'label']).reset_index(drop=True)
df = df.drop_duplicates(subset=['body']).reset_index(drop=True)

print("Total rows:", len(df))
print("Unique messages:", df['body'].nunique())


Total rows: 1890
Unique messages: 1890


In [140]:
y = (df['label'].str.lower().str.strip() == 'phishing').astype(int)
X = df['body'].fillna('')

In [141]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [142]:
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1,2))
X_train_proc = tfidf.fit_transform(X_train)
X_test_proc  = tfidf.transform(X_test)

In [143]:
print("\nTraining Random Forest Model...")
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X_train_proc, y_train)
print("✅ Model trained successfully!")

# Optional: Evaluation Check to see how well it performs on test split
y_pred = random_forest_model.predict(X_test_proc)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")


Training Random Forest Model...
✅ Model trained successfully!
Test Accuracy: 1.0000


In [144]:
def predict_new_message(phone_number, message_body):
    # Text ko transform karo TF-IDF matrix mein
    body_vec = tfidf.transform([message_body])
    
    # Direct RandomForest use karo yahan variable name badal kar
    probability = float(random_forest_model.predict_proba(body_vec)[0][1])
    
    # Risk level smoothing (1 to 5)
    risk_level = min(int(probability * 5) + 1, 5)

    return {
        'Phone'           : phone_number,
        'Is Fraud'        : probability > 0.5,
        'Risk Level (1-5)': risk_level,
        'Confidence'      : f"{probability * 100:.1f}%"
    }

In [147]:
# ── 8. TEST ──────────────────────────────────────────────────
print("\n[TEST 1] Fraud message:")
print(predict_new_message(
    "923001234567",
    "URGENT: Your bank account is suspended. "
))

print("\n[TEST 2] Normal message:")
print(predict_new_message(
    "923219876543",
    "https://www.google.com/search?sca_esv=6fef7d0a1afa4d8f&sxsrf=ANbL-n7MREypSpL83ddcTuGPXbVBq-ln8g:1780764596415&udm=2&fbs=ADc_l-aN0CWEZBOHjofHoaMMDiKpaEWjvZ2Py1XXV8d8KvlI3sbM0Xv-BZKE_VrZb6-djVhOtEikQxVPaqfTMUjZRiWtc4eIxdwaGq8rf74gkoZQnfxrgsCssjzlM59qezx9pUwvBV1BjT8uWad5IYX7mEST_rjvD31q-Hxi5Tdhku1yVE_6iBjYBzZTAPtOmUxlCYHW-sHV&q=WhatsApp+uses+a+Blue+Tick+(Verified+Badge)&sa=X&ved=2ahUKEwjAr8qJifOUAxW8hP0HHWk4GggQtKgLegQIFRAB&biw=1280&bih=585&dpr=1.5"
))


[TEST 1] Fraud message:
{'Phone': '923001234567', 'Is Fraud': True, 'Risk Level (1-5)': 3, 'Confidence': '54.0%'}

[TEST 2] Normal message:
{'Phone': '923219876543', 'Is Fraud': False, 'Risk Level (1-5)': 1, 'Confidence': '9.0%'}
